In [1]:
import pandas as pd
import numpy as np
import time
import itertools
from sklearn.pipeline import make_pipeline

### Информация о пассажирах

* Пол пассажира (женский/мужской)
* Тип клиента Тип пассажира (лояльный/нелояльный)
* Возраст Возраст пассажира в годах (числовой)
* Тип путешествия
* Класс полета Бизнес/Эко/Эко Плюс
* Расстояние полета

### Рейтинги качества обслуживания

* Услуга Wi-Fi в полете
* Удобное время отправления/прибытия
* Простота онлайн-бронирования
* Расположение ворот
* Продукты питания
* Онлайн-интернат
* Комфорт сиденья
* Развлечения в полете
* Бортовое обслуживание
* Обслуживание комнаты для ног
* Обработка багажа
* Услуга регистрации
* Чистота

### Рейтинги задержки

* Удовлетворенность задержкой вылета (1–5 или 0, если не оценено)
* Удовлетворенность задержкой прибытия (1–5 или 0, если не оценено)

### Целевая переменная
* удовлетворение	Удовлетворенность пассажиров (Правда = Удовлетворенность, Ложь = Не Удовлетворенность)

In [2]:
df = pd.read_csv('avia_linear.csv', sep=',', encoding='mac_cyrillic')

columns = df.columns
print(f"Всего пропусков: {df.isnull().sum().sum()}")
print(f"Всего (строк, столбцов): {df.shape}")

Всего пропусков: 393
Всего (строк, столбцов): (129880, 22)


### Анализ пропусков

Количество пропусков (393) несоизмеримо мало с количеством данных (129.880), что составляет около 0.3% от общего количества, поэтому данные строки можно удалить и сразу преобразовать качественные значения в количественные

In [3]:
df = df.dropna(subset=['Arrival Delay in Minutes'])

df['Gender'] = np.where(df['Gender'] == 'Male', 1, 0)
df['Customer Type'] = np.where(df['Customer Type'] == 'Loyal Customer', 1, 0)
df['Type of Travel'] = np.where(df['Type of Travel'] == 'Business travel', 1, 0)
df['satisfaction'] = np.where(df['satisfaction'] == True, 1, 0)
df['Class'] = np.where(
    df['Class'] == 'Business', 2, 
    np.where(df['Class'] == 'Eco Plus', 1, 0)
)

# делим на признаки (x) и целевую переменную (y)
x = df.drop(columns=['Arrival Delay in Minutes']).to_numpy(dtype=float)
y = df['Arrival Delay in Minutes'].to_numpy(dtype=float)

print(f"Размерность х: {x.shape}")
print(f"Размерность y: {y.shape}")

Размерность х: (129487, 21)
Размерность y: (129487,)


In [4]:
def train_test_split(x, y, test_size, random_state=42):
    np.random.seed(random_state)
    indices = np.random.permutation(len(x))
    test_set_size = int(len(x) * test_size)
    
    test_indices = indices[:test_set_size]
    train_indices = indices[test_set_size:]
    
    return x[train_indices], x[test_indices], y[train_indices], y[test_indices]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
print(len(x_train), len(x_test))

103590 25897


In [5]:
class CustomStandardScaler:
    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        self.std[self.std == 0] = 1.0
        return self

    def transform(self, X): # маштабирование данных
        return (X - self.mean) / self.std

    def fit_transform(self, X):
        return self.fit(X).transform(X)


scaler = CustomStandardScaler()
x_train_scaler = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)

In [6]:
class CustomLinearRegression_Analytical:
    def __init__(self, fit_intercept=True):
        self.fit_intercept = fit_intercept # флаг, указывающий, нужно ли добавлять свободный член w_0 в модель
        self.weights = None # массив весов

    def fit(self, X, y):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([ones, X]) # приклеивание вектора из единиц для свободного члена
        
        # Формула аналитического решения (OLS): w = (X^T * X)^(-1) * X^T * y
        self.weights = np.linalg.inv(X.T @ X) @ X.T @ y
        return self

    def predict(self, X):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([ones, X])
            
        return X @ self.weights

class CustomLinearRegression_Gradient:
    def __init__(self, learning_velocity=0.01, n=1000, fit_intercept=True):
        self.learning_velocity = learning_velocity
        self.n = n
        self.fit_intercept = fit_intercept
        self.weights = None

    def fit(self, X, y):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([ones, X])
        
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)

        for _ in range(self.n):
            gradient = (2 / n_samples) * (X.T @ (X @ self.weights - y))
            self.weights -= self.learning_velocity * gradient
        
        return self

    def predict(self, X):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([ones, X])
        
        return X @ self.weights

In [7]:
def smse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

In [8]:
model_linear = CustomLinearRegression_Analytical(fit_intercept=True)
model_gradient = CustomLinearRegression_Gradient(learning_velocity=0.1, n=10, fit_intercept=True)

start_time_linear = time.time()
model_linear.fit(x_train_scaler, y_train)
time_linear = time.time() - start_time_linear

start_time_gradient = time.time()
model_gradient.fit(x_train_scaler, y_train)
time_gradient = time.time() - start_time_gradient

y_train_pred_linear = model_linear.predict(x_train_scaler)
y_test_pred_linear = model_linear.predict(x_test_scaler)

y_train_pred_gradient = model_gradient.predict(x_train_scaler)
y_test_pred_gradient = model_gradient.predict(x_test_scaler)

print(f"Время работы по аналитическому решению (Linear Regression): {time_linear}")
print(f"Время работы по градиентному спуску (Gradient Descent): {time_gradient}")

print()
print("Метрики на тестовой выборке (Test) Linear Regression")
print(f"RMSE: {smse(y_test, y_test_pred_linear):.4f}")
print(f"MAE:  {mae(y_test, y_test_pred_linear):.4f}")
print(f"R2:   {r2(y_test, y_test_pred_linear):.4f}")

print()
print("Метрики на тестовой выборке (Test) Gradient Descent")
print(f"RMSE: {smse(y_test, y_test_pred_gradient):.4f}")
print(f"MAE:  {mae(y_test, y_test_pred_gradient):.4f}")
print(f"R2:   {r2(y_test, y_test_pred_gradient):.4f}")

Время работы по аналитическому решению (Linear Regression): 0.017837047576904297
Время работы по градиентному спуску (Gradient Descent): 0.045252084732055664

Метрики на тестовой выборке (Test) Linear Regression
RMSE: 9.9777
MAE:  5.3320
R2:   0.9419

Метрики на тестовой выборке (Test) Gradient Descent
RMSE: 10.9789
MAE:  5.5033
R2:   0.9297


In [ ]:
class CustomPolynomialFeatures:
    def __init__(self, degree, fit_intercept=True):
        self.fit_intercept = fit_intercept # флаг, указывающий, нужно ли добавлять свободный член w_0 в модель
        self.degree = degree

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if self.degree == 1:
            return X
        
        n_strok, n_stolbcov = X.shape
        poly = []

        if self.fit_intercept:
            poly.append(np.ones((n_strok, 1))) # добавление w0
        
        for len in range(1, self.degree + 1):
            for values in itertools.combinations_with_replacement(range(n_stolbcov), len): # группировка значений с повторениями (с повторениями = х1х1, ..., х2х2, ...; здесь len==2)
                pol = np.prod(X[:, values], axis=1, keepdims=True)
                poly.append(pol)
        return np.hstack(poly)
    
    def fit_transform(self, X, y=):
        return self.fit(X, y).transform(X)

poly = CustomPolynomialFeatures(degree=2)

x_train_poly = poly.fit_transform(x_train)

In [ ]:
# --- Построение графика R² от степени полинома ---
degrees_to_test = range(1, 12) # Протестируем степени от 1 до 15
train_scores = []
test_scores = []

for degree in degrees_to_test:
    poly = CustomPolynomialFeatures(degree=degree, fit_intercept=True)
    x_train_poly = poly.fit_transform(x_train_scaler)
    x_test_poly = poly.transform(x_test_scaler)
    
    model = CustomLinearRegression_Analytical(fit_intercept=False)
    model.fit(x_train_poly, y_train)
    
    y_train_pred = model.predict(x_train_poly)
    y_test_pred = model.predict(x_test_poly)
    
    train_scores.append(r2(y_train, y_train_pred))
    test_scores.append(r2(y_test, y_test_pred))

    print(f"Степень {degree}: R² на обучении = {train_scores[-1]:.4f}, R² на тесте = {test_scores[-1]:.4f}")
    
plt.figure(figsize=(10, 6))
plt.plot(degrees_to_test, train_scores, 'bo-', label='Train R²')
plt.plot(degrees_to_test, test_scores, 'ro-', label='Test R²')
plt.xlabel('Степень полинома')
plt.ylabel('R²')
plt.title('R² на обучающей и тестовой выборках vs степень полинома')
plt.legend()
plt.grid(True)

# Отметим максимум на тестовой кривой
best_degree_idx = np.argmax(test_scores)
best_degree = degrees_to_test[best_degree_idx]
plt.axvline(best_degree, linestyle='--', color='gray', label=f'Оптимальная степень ≈ {best_degree}')
plt.legend()
plt.ylim(min(train_scores + test_scores) - 0.1, 1.1) # Настроить масштаб оси Y
plt.xticks(degrees_to_test[::2]) # Показать не все тики на оси X
plt.show()

print(f"\nМаксимальный R² на тесте (R²={test_scores[best_degree_idx]:.2f}) достигается при степени {best_degree}")
# Максимальный R² на тесте (R²=0.65) достигается при степени 5